# `klygo.config` (`validation.py`): Lightweight schema and key validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Validation.ipynb)


Validate required nested key paths, expected types or type tuples, callable predicates, and literal values.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-4xytwt1q


## Jump to an API

[`validate`](#validate)

<a id="validate"></a>

### `config.validate`

Validate required paths and optional expected value contracts.

**Signature:** `config.validate(config_data: Mapping[str, Any] | box.Box, required_keys: Sequence[str] | Mapping[str, Any] | None = None) -> bool`

In [2]:
candidate = {"model": {"name": "yolov8n", "batch": 16, "lr": 0.01}}
print("Sequence check:", config.validate(candidate, ["model.name", "model.batch"]))
schema = {
    "model.name": str,
    "model.batch": lambda x: isinstance(x, int) and x > 0,
    "model.lr": (int, float),
}
print("Schema check:", config.validate(candidate, schema))

Sequence check: True
Schema check: True


In [3]:
# Error examples: missing required key and failed type/predicate check
for bad_schema in (["model.missing"], {"model.batch": str}, {"model.batch": lambda x: x >= 128}):
    try:
        config.validate(candidate, bad_schema)
    except ValueError as exc:
        print("ValueError ->", exc)

ValueError -> missing required config key: 'model.missing'
ValueError -> config key 'model.batch' must be <class 'str'>, got int
ValueError -> config key 'model.batch' failed validation


## Put it together

In [4]:
candidate = {"model": {"name": "resnet50", "lr": 0.001}, "seed": 42}
is_valid = config.validate(
    candidate,
    {"model.name": str, "model.lr": (int, float), "seed": int},
)
print("Validated candidate:", is_valid)

Validated candidate: True


## Remember

- `validate()` checks required dot-paths (via a list of keys) or expected types (via a `{key_path: type}` mapping, rejecting `bool` when `int` is requested unless `bool` is explicitly allowed) and returns `True` when valid.
- Next in `klygo.config`: continue with [Document_Config_Config.ipynb](Document_Config_Config.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).